### For computing solvated entropies, you will need at least two strucures - the solute and the solvent.

In [1]:
from ase.build import molecule
from pymatgen.io.ase import AseAtomsAdaptor

water_struct = AseAtomsAdaptor.get_structure(molecule("H2O", vacuum=10.0))
acetic_acid_struct = AseAtomsAdaptor.get_structure(molecule("CH3COOH", vacuum=10.0))

### The next step will be to convert these into my `StructureVolume` object (acts like a `Structure` but can do some volume computation stuff)

In [2]:
from JDFTxFreeNrg.volume import StructureVolume

water_sv = StructureVolume.from_structure(water_struct)
acetic_acid_sv = StructureVolume.from_structure(acetic_acid_struct)

### By default, these `StructureVolume`'s will be initialized to use monte-carlo integration. Volumes for the total structure can be retrieved by the `get_volume` method. The accuracy of this integration can be controlled with `npoints` or `grid_spacing` (which does nothing for MC integration, but also is not implemented for mesh yet)

In [3]:
water_vol = water_sv.get_volume(npoints=100000)
acetic_acid_vol = acetic_acid_sv.get_volume(npoints=100000)

### Usable values might require a bit of time to compute. We can avoid re-calculating redundant volumes by setting cache directories for our structures upon initialization

In [4]:
from os import getcwd
from pathlib import Path

h2o_cache = Path(getcwd()) / "data" / "H2O_cache"
acetic_acid_cache = Path(getcwd()) / "data" / "CH3COOH_cache"

water_sv = StructureVolume.from_structure(water_struct, cache_parent=h2o_cache)
acetic_acid_sv = StructureVolume.from_structure(acetic_acid_struct, cache_parent=acetic_acid_cache)
water_sv.clear_cache()
acetic_acid_sv.clear_cache()

### Now requesting the same integration a second time will recall the first value computed.

In [5]:
from time import time

start = time()
water_vol = water_sv.get_volume(npoints=1e5)
end = time()
print(f"Initial computation time: {end - start} seconds")

start = time()
water_vol = water_sv.get_volume(npoints=1e5)
end = time()
print(f"Cached computation time: {end - start} seconds")

Initial computation time: 1.3651738166809082 seconds
Cached computation time: 5.2928924560546875e-05 seconds


### Now changing the accuracy parameter will trigger a reevaluation for that level of accuracy

In [ ]:
start = time()
water_vol = water_sv.get_volume(npoints=1e5)
end = time()
print(f"Computation time with n = 100000: {end - start} seconds")

start = time()
water_vol = water_sv.get_volume(npoints=1e5 + 1)
end = time()
print(f"Computation time with n = 100001: {end - start} seconds")

Computation time with n = 100000: 8.893013000488281e-05 seconds
Computation time with n = 50000: 1.263700008392334 seconds


### Not specifying the level of accuracy will automatically grab the most expensive one generated

In [7]:
start = time()
water_vol = water_sv.get_volume()
end = time()
print(f"Computation time with default npoints: {end - start} seconds")

Computation time with default npoints: 8.606910705566406e-05 seconds


### We may not be interested in the total VdW volume of the structure, but that of a substructure of a structure (ie a molecule desorbed from a slab). We can specify the substructure by the atom indices of the substructure when getting our volume

In [9]:
sub_acetic_acid_vol = acetic_acid_sv.get_volume(idcs=[0,1,2], npoints=1e5)

### We can also specify the method of integration upon initialization of the `StructureVolume` object. Options are "MC", "Mesh", and "PyVol" (case insensitive)

In [15]:
water_sv = StructureVolume.from_structure(water_struct, cache_parent=h2o_cache, method="Mesh")
acetic_acid_sv = StructureVolume.from_structure(acetic_acid_struct, cache_parent=acetic_acid_cache, method="Mesh")
water_sv.clear_cache()
acetic_acid_sv.clear_cache()

### Note for mesh integration, the "npoints" parameter (now representing number of voxels in our mesh) is not 1:1 in time cost with MC grid (where "npoints" specified the number of random points sampled). On my machine the cost of mesh integration is ~100x cheaper for a given npoints, so ive changed "npoints" to 1e7. However if you were to put in "1e5" again, the caching system would compute a new value since it only has MC values archived for 1e5 points. 

In [16]:
start = time()
water_vol = water_sv.get_volume(npoints=1e7)
end = time()

### For all the solvation entropy equations we will need three values - the molarity of our solvent, and the VdW volumes of our solvent and solute. The molarity is needed for computing "vfree" (free space per solvent molecule)

In [20]:
from JDFTxFreeNrg.solv_entropy import get_vfree

water_vol = water_sv.get_volume(npoints=1e7)
water_vfree = get_vfree(water_vol, 55.5)
acetic_acid_vol = acetic_acid_sv.get_volume(npoints=1e7)


### `get_solv_entropy_trans` requires the above values, along with a `Structure` for the solute (but also accepts a `StructureVolume`), the temperature (in K), and the dimensionality of the free translations (most likely 3, but you can change to 1 or 2 for partially restricted solutes)

In [ ]:
from JDFTxFreeNrg.solv_entropy import get_solv_entropy_trans, J_to_eV
from JDFTxFreeNrg.standard import get_entropy_trans, get_ideal_gas_vol
import numpy as np

T = 300.
solv_acetic_acid_entropy_trans = get_solv_entropy_trans(acetic_acid_sv, acetic_acid_vol, water_vol, water_vfree, T, d=3)
# print(f"Solvation entropy transfer of acetic acid at {T} K: {solv_acetic_acid_entropy_trans} eV/K")
# gas_acetic_acid_entropy_trans = get_entropy_trans(np.sum([site.specie.atomic_mass for site in acetic_acid_sv.sites]), T, get_ideal_gas_vol(1.0, T), d=3) * J_to_eV
# print(f"Ideal gas entropy transfer of acetic acid at {T} K: {gas_acetic_acid_entropy_trans} eV/K")

Solvation entropy transfer of acetic acid at 300.0 K: 0.0012081364394594602 eV/K
Ideal gas entropy transfer of acetic acid at 300.0 K: 0.0016577509503857115 eV/K
(72.88% of ideal gas value)


### `get_solv_entropy_rot` doesn't require the solvent volume, only the free volume

In [28]:
from JDFTxFreeNrg.solv_entropy import get_solv_entropy_rot

solv_acetic_acid_entropy_rot = get_solv_entropy_rot(acetic_acid_sv, acetic_acid_vol, water_vfree, T)
print(f"Solvation entropy rotational of acetic acid at {T} K: {solv_acetic_acid_entropy_rot} eV/K")

Solvation entropy rotational of acetic acid at 300.0 K: 0.0008942824108689126 eV/K
